In [107]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [108]:
df = pd.read_excel('exerc3.xlsx', index_col=0)

In [109]:
df

,loja1,loja2,loja3,loja4
deposito,,,,
dep1,5,4,6,5
dep2,3,6,4,4
dep3,4,3,3,2


In [110]:
lojas = df.columns.tolist()
depositos = df.index.tolist()

In [111]:
df.loc['dep3']['loja4']

2

In [112]:
df.stack().index.tolist()

[('dep1', 'loja1'),
 ('dep1', 'loja2'),
 ('dep1', 'loja3'),
 ('dep1', 'loja4'),
 ('dep2', 'loja1'),
 ('dep2', 'loja2'),
 ('dep2', 'loja3'),
 ('dep2', 'loja4'),
 ('dep3', 'loja1'),
 ('dep3', 'loja2'),
 ('dep3', 'loja3'),
 ('dep3', 'loja4')]

In [113]:
#modelo

model = pyo.ConcreteModel()

model.lojas = pyo.Set(initialize=lojas)
model.depositos = pyo.Set(initialize=depositos)
model.oferta = pyo.Param(model.depositos, initialize=40)
model.demanda = pyo.Param(model.lojas, initialize=dict(zip(lojas, [20, 25, 30, 35])))
model.relacao = pyo.Set(initialize=df.stack().index.tolist(),dimen=2)
model.custos = pyo.Param(model.depositos, model.lojas, initialize=df.stack().to_dict())
model.x = pyo.Var(model.depositos, model.lojas, domain=pyo.NonNegativeIntegers)

In [114]:
#objetivo

def obj_rule(model):
    return sum(model.custos[r]*model.x[r] for r in model.relacao)
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)


In [115]:
#restrições

def restricao_oferta(model, d):
    saida  = sum(model.x[c] for c in model.relacao if c[0] == d)
    entrada = sum(model.x[c] for c in model.relacao if c[1] == d)
    return saida - entrada <= model.oferta[d]
model.restricao_oferta = pyo.Constraint(model.depositos, rule=restricao_oferta)

def restricao_demanda(model, l):
    entrada = sum(model.x[c] for c in model.relacao if c[1] == l)
    saida = sum(model.x[c] for c in model.relacao if c[0] == l)
    return entrada - saida >= model.demanda[l]
model.restricao_demanda = pyo.Constraint(model.lojas, rule=restricao_demanda)

In [116]:
opt = pyo.SolverFactory('cplex')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpedz482xd.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpce9toxue.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpce9toxue.pyomo.lp
Objective sense      : Minimize
Variables            :      12  [General Integer: 12]
Objective nonzeros   :      12
Linear constraints   :       7  [Less: 3,  Greater: 4]
  Nonzeros           :      24
  RHS nonzeros       :       7

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 2

In [117]:
model.pprint()

3 Set Declarations
    depositos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'dep1', 'dep2', 'dep3'}
    lojas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'loja1', 'loja2', 'loja3', 'loja4'}
    relacao : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     2 :    Any :   12 : {('dep1', 'loja1'), ('dep1', 'loja2'), ('dep1', 'loja3'), ('dep1', 'loja4'), ('dep2', 'loja1'), ('dep2', 'loja2'), ('dep2', 'loja3'), ('dep2', 'loja4'), ('dep3', 'loja1'), ('dep3', 'loja2'), ('dep3', 'loja3'), ('dep3', 'loja4')}

3 Param Declarations
    custos : Size=12, Index=depositos*lojas, Domain=Any, Default=None, Mutable=False
        Key               : Value
        ('dep1', 'loja1') :     5
        ('dep1', 'loja2') :     4
        ('dep1', 'loja3') :     6
        ('dep1', 'loja4') :     5

In [123]:
for d,l in model.x:
    print(d, " para ", l, ", Quantidade: ", model.x[d,l].value)

dep1  para  loja1 , Quantidade:  -0.0
dep1  para  loja2 , Quantidade:  25.0
dep1  para  loja3 , Quantidade:  5.0
dep1  para  loja4 , Quantidade:  -0.0
dep2  para  loja1 , Quantidade:  20.0
dep2  para  loja2 , Quantidade:  -0.0
dep2  para  loja3 , Quantidade:  20.0
dep2  para  loja4 , Quantidade:  -0.0
dep3  para  loja1 , Quantidade:  -0.0
dep3  para  loja2 , Quantidade:  -0.0
dep3  para  loja3 , Quantidade:  5.0
dep3  para  loja4 , Quantidade:  35.0
